<!--
Copyright 2026 Yaroslav Mariukha
SPDX-License-Identifier: RPL-1.5

Unless explicitly acquired and licensed from Licensor under another license,
the contents of this file are subject to the Reciprocal Public License ("RPL")
Version 1.5, or subsequent versions as allowed by the RPL, and You may not copy
or use this file in either source code or executable form, except in compliance
with the terms and conditions of the RPL.

All software distributed under the RPL is provided strictly on an "AS IS"
basis, WITHOUT WARRANTY OF ANY KIND, EITHER EXPRESS OR IMPLIED, AND LICENSOR
HEREBY DISCLAIMS ALL SUCH WARRANTIES, INCLUDING WITHOUT LIMITATION, ANY
WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE, QUIET
ENJOYMENT, OR NON-INFRINGEMENT. See the RPL for specific language governing
rights and limitations under the RPL.
-->


# Intel VIP Verification Components

This tutorial connects Intel VIP packet objects to a pyuvm verification environment. Object construction and reference-model code are ordinary Python; agents, drivers, monitors, and scoreboards run inside a cocotb simulation.

## 1. Verification architecture

```text
VIPSequence → VIPDriver → Avalon-ST source → DUT → Avalon-ST sink
                            │                    │
                       source monitor       sink monitor
                            │                    │
                            │                    │
                            └──→ custom scoreboard ←──┘
                                  │
                            IP behavior model
```

The packet codec remains independent of simulation. `VIPDriver` serializes packet objects, while `VIPMonitor` reconstructs them and publishes them through a pyuvm analysis port.

In [1]:
from fpga_verification.protocols.avalon_st.intel_video import (
    IntelVIPFrameCodec,
    VIPControlPacket,
    VIPUserPacket,
)
from fpga_verification.sim.agents import (
    VIPAgent,
    VIPDriver,
    VIPItem,
    VIPMonitor,
    VIPSequence,
)
from cocotbext.avalon import AvalonSTBus
from fpga_verification.sim.scoreboards import (
    AnalysisImp,
    BaseVIPScoreboard,
    CheckMode,
    PacketExpectation,
    UserPacketPolicy,
)
from fpga_verification.video import FrameSize, ImageGenerator, VideoFormat

## 2. VIPItem and VIPSequence

`VIPItem` is a pyuvm sequence item containing exactly one complete VIP packet. `VIPSequence.from_packets()` converts an ordered packet list into items. Its `body()` sends every item through a sequencer.

A frame is normally sent as separate user, control, and video items so that packet boundaries remain visible to the driver and monitor.

In [2]:
fmt = VideoFormat(
    bits_per_color=8,
    number_of_color_planes=3,
    pixels_in_parallel=1,
)
size = FrameSize(width=4, height=2)
frame = ImageGenerator(fmt).horizontal_ramp(size)
packets = IntelVIPFrameCodec(fmt).frame_to_packets(
    frame,
    size,
    user_packets=[VIPUserPacket(1, [0xA, 0xB])],
)

item = VIPItem.from_packet(packets[0])
sequence = VIPSequence.from_packets(packets, name="input_frame")

print(type(item.to_vip_packet()).__name__)
print([type(sequence_item.packet).__name__ for sequence_item in sequence.items])

VIPUserPacket
['VIPUserPacket', 'VIPControlPacket', 'VIPVideoPacket']


## 3. VIPDriver and VIPMonitor

`VIPDriver` receives `VIPItem` objects, calls `packet.to_symbols()`, and sends one `AvalonSTFrame` per packet. It is normally created by an active `VIPAgent`.

`VIPMonitor` receives complete Avalon-ST frames, decodes them with `vip_packet_from_symbols()`, checks ordering with `VIPProtocolChecker`, and publishes packet objects. With `drive_ready=False` it observes passively; with `drive_ready=True` it acts as a sink and controls backpressure.

In [3]:
def make_passive_monitor(parent, dut, fmt):
    return VIPMonitor(
        "vip_tap",
        parent,
        bus=AvalonSTBus.from_prefix(dut, "vip_out"),
        clock=dut.clk,
        reset=dut.reset,
        fmt=fmt,
        drive_ready=False,
        packet_logging=True,
    )

## 4. VIPAgent

`VIPAgent` assembles the source driver, sequencer, source monitor, and/or sink monitor. Provide either bus or both buses:

- `source_bus` is driven toward the DUT and also monitored;
- `sink_bus` is observed after the DUT and, in an active agent, drives `ready`;
- a passive agent only observes existing traffic;
- `randomize=True` introduces source pauses and sink backpressure;
- `source_analysis_port` and `sink_analysis_port` publish decoded packets.

In [4]:
def make_active_agent(parent, dut, fmt):
    return VIPAgent(
        "vip_agent",
        parent,
        clock=dut.clk,
        reset=dut.reset,
        source_bus=AvalonSTBus.from_prefix(dut, "vip_in"),
        sink_bus=AvalonSTBus.from_prefix(dut, "vip_out"),
        source_fmt=fmt,
        sink_fmt=fmt,
        randomize=True,
        packet_logging=True,
    )

# During a pyuvm run phase:
# await sequence.start(agent.sequencer)

## 5. PacketExpectation and behavior models

A custom scoreboard consumes input packets and control transactions and produces one `PacketExpectation` for each expected output packet. `CheckMode.EXACT` compares complete decoded content. `CheckMode.SHAPE` checks packet type and observable geometry when output content is intentionally undefined.

Keep pure frame/packet transforms in a functional model and register/reset/temporal state in a behavior model owned by the IP-specific scoreboard. `UserPacketPolicy.DROP` is the default. Passthrough IPs create an exact expectation; transforming IPs create the transformed expectation.

In [5]:
expectations = [
    PacketExpectation(packet, check=CheckMode.EXACT)
    for packet in packets
]
print([type(item.packet).__name__ for item in expectations])

['VIPUserPacket', 'VIPControlPacket', 'VIPVideoPacket']


## 6. AnalysisImp and BaseVIPScoreboard

`AnalysisImp` adapts a pyuvm analysis export to a Python callback. `BaseVIPScoreboard` provides three protocol-facing exports:

- `data_in_export` invokes `process_input_packet()` when `source_fmt` is present;
- `data_out_export` compares output packets against queued expectations;
- `control_export` invokes `process_control_transaction()`.

Reset assertion aborts pending expectations and calls `on_reset()` without hiding a sticky mismatch. `drain()` waits for an empty queue and an optional quiet output window.

In [6]:
class IdentityScoreboard(BaseVIPScoreboard):
    user_packet_policy = UserPacketPolicy.PASSTHROUGH

    def process_input_packet(self, packet):
        self.add_expectation(PacketExpectation(packet))

    def process_control_transaction(self, transaction):
        pass

    def on_reset(self):
        pass


def connect_vip_environment(agent, scoreboard):
    agent.source_analysis_port.connect(scoreboard.data_in_export)
    agent.sink_analysis_port.connect(scoreboard.data_out_export)


# Typical pyuvm build phase:
# env.agent = make_active_agent(env, cocotb.top, fmt)
# env.scoreboard = IdentityScoreboard(
#     "scoreboard", env, source_fmt=fmt, sink_fmt=fmt,
#     clock=cocotb.top.clk, reset=cocotb.top.reset, quiet_cycles=2,
# )
# connect_vip_environment(env.agent, env.scoreboard)

## 7. Lifecycle and cleanup

Use `agent.set_packet_logging()` to change packet summaries and `agent.set_randomize()` to enable or disable pauses after construction. `cancel_bfms()` stops background bus tasks; `clear_bfms()` clears queued traffic and resets monitor protocol state. Reset also clears pending scoreboard expectations while preserving earlier failures. End a test with `await scoreboard.drain(timeout, quiet_cycles=...)`.

Generated DUTs use one resolved `ComponentConfig`: the host selects a named case through `FPGA_VERIFICATION_CONFIG_CASE`, `run_intel_component_test()` passes its JSON through `extra_env`, and cocotb restores it only with `load_runtime_config(TestConfig)`. Missing, stale, or malformed runtime configuration is an error.